# Saved-model prediction example

> **SIMULATION ONLY.** The example is a held-out synthetic row and does not represent a real crop or sensor reading.

This notebook loads saved artifacts; it does not retrain. It prints current status probabilities and next-inspection risk. The model requires current environment and all nine MQ channels plus the history fields listed below. If a real reading is missing a required field, the prediction function raises a clear error rather than filling it in.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "artifacts").exists():
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"


In [2]:
import sys
sys.path.insert(0, str(ROOT))
from crop_prototype.inference import load_prediction_models, predict_payload

models = load_prediction_models(ARTIFACTS)
status_model = models["status"]
status_features = list(status_model.named_steps["prep"].feature_names_in_)
required = [name for name in status_features if name not in {"crop", "grain_moisture_pct_SIM", "lot_mass_kg_SIM"}]
required += ["next_inspection_interval_days"]
print("Required prediction inputs:")
print("\n".join("- " + name for name in ["crop"] + required))
print("\nMQ values are response indices, not gas concentrations. Environmental units follow the device field names; history fields must be calculated through the current reading.")


Required prediction inputs:
- crop
- storage_age_days
- Temperature(°C)
- Humidity(%)
- Pressure(hPa)
- AQI
- Gas (kOhms)
- Altitude(m)
- DewPoint(°C)
- HeatIndex(°C)
- past_mean_temp_C_SIM
- past_mean_rh_pct_SIM
- past_hot_degree_days_SIM
- MQ2_response_index_SIM
- MQ3_response_index_SIM
- MQ4_response_index_SIM
- MQ5_response_index_SIM
- MQ6_response_index_SIM
- MQ7_response_index_SIM
- MQ8_response_index_SIM
- MQ9_response_index_SIM
- MQ135_response_index_SIM
- next_inspection_interval_days

MQ values are response indices, not gas concentrations. Environmental units follow the device field names; history fields must be calculated through the current reading.


In [3]:
demo_path = ROOT / "examples" / "simulated_prediction_input.json"
payload = json.loads(demo_path.read_text())
prediction = predict_payload(payload, ARTIFACTS, models=models)
print(json.dumps(prediction, indent=2))


{
  "label": "SIMULATION ONLY",
  "crop": "maize",
  "storage_age_days": 0.0,
  "predicted_status": "acceptable",
  "status_probabilities": {
    "acceptable": 0.9993624091148376,
    "warning": 0.0005873488262295723,
    "spoiled": 5.020654134568758e-05
  },
  "next_inspection_interval_days": 7.0,
  "probability_next_check_leaves_acceptable_state": 0.004692844812262783,
  "early_warning_alert_threshold": 0.09747646188292124,
  "early_warning_alert": false,
  "probability_first_spoiled_inspection_by_next_check": 7.112255843821913e-05,
  "remaining_days_estimate": null,
  "warnings": [
    "Pressure(hPa) is outside the 1st-99th percentile range in the maize training simulation",
    "MQ5_response_index_SIM is outside the 1st-99th percentile range in the maize training simulation"
  ],
  "note": "SIMULATION ONLY. These saved models reconstruct simulator assumptions. Probabilities are conditional on the supplied current/history inputs and next-check interval; no continuous remaining-days 

## Entering another reading

Copy the example JSON schema into a new payload and replace values with readings from the same crop lot. Keep units consistent. Do not enter the inspection answer or a label. Compute the history fields from that lot's readings through the current time. Optional grain moisture and lot mass may be omitted; required inputs may not. The saved model was trained on simulation, so a real sensor reading can be outside its training range and the prediction is not validated.

In [4]:
# Example input template only. Replace every None with a measured value before calling predict_payload.
# The template intentionally does not make a prediction with invented sensor values.
reading_template = {
    "crop": None,  # wheat, maize, potato, onion, or spinach
    "storage_age_days": None,
    "Temperature(\u00b0C)": None,
    "Humidity(%)": None,
    "Pressure(hPa)": None,
    "AQI": None,
    "Gas (kOhms)": None,
    "Altitude(m)": None,
    "DewPoint(\u00b0C)": None,
    "HeatIndex(\u00b0C)": None,
    "MQ2_response_index_SIM": None,
    "MQ3_response_index_SIM": None,
    "MQ4_response_index_SIM": None,
    "MQ5_response_index_SIM": None,
    "MQ6_response_index_SIM": None,
    "MQ7_response_index_SIM": None,
    "MQ8_response_index_SIM": None,
    "MQ9_response_index_SIM": None,
    "MQ135_response_index_SIM": None,
    "past_mean_temp_C_SIM": None,
    "past_mean_rh_pct_SIM": None,
    "past_hot_degree_days_SIM": None,
    "next_inspection_interval_days": None,
}
# Once populated from real, paired measurements:
# prediction = predict_payload(reading_template, ARTIFACTS)
